# LoRAForge — Phase 2 training on a fresh T4

This refreshed notebook performs only the safe validation workflow: clone a clean checkout, verify Phase 1 evidence, train two frozen QLoRA epochs, select on validation, and freeze the selection. **It does not load the publisher test split.**

Do not use **Run all**. Run each code cell in order. Stop after the freeze-and-verify cell.

In [ ]:
from pathlib import Path
from time import time

BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = BASE / 'loraforge-llm-phase2-run'
if (REPO / '.git').exists():
    !git -C {REPO} pull --ff-only
else:
    if REPO.exists():
        backup = REPO.with_name(f'{REPO.name}-backup-{int(time())}')
        REPO.rename(backup)
        print(f'Preserved stale folder at {backup}')
    !git clone --depth 1 https://github.com/mghadia1/loraforge-llm.git {REPO}
assert (REPO / '.git').exists(), 'clean clone failed; inspect the git output above'
%cd {REPO}
!python -m pip install -q -e ".[gpu]"


In [ ]:
import sys, site
site.main()  # an editable install adds a .pth the running kernel has not read
sys.path.insert(0, str(REPO / 'src'))
import loraforge
print('loraforge importable from', loraforge.__file__)


In [ ]:
import json, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
gpu_name = torch.cuda.get_device_name(0)
assert 'T4' in gpu_name, f'Frozen evidence run expects a T4, found {gpu_name}'
assert Path('outputs/base-validation.json').exists(), 'run phase 1 and restore its artifact first'
assert Path('outputs/qlora-setup.json').exists(), 'run phase 1 and restore its artifact first'
print({'gpu': gpu_name, 'torch': torch.__version__})


In [ ]:
!python -m pytest -q


## Step 4 — two frozen epochs

`train_qlora` re-scores the untuned base with the adapter disabled first and refuses to
continue unless it reproduces the phase-one baseline macro-F1. It then trains the frozen
two epochs, saves and scores a checkpoint after each one, and selects the higher validation
macro-F1 (an exact tie goes to the earlier epoch). Loss sees only the answer code and EOS.


In [ ]:
from loraforge.config import default_config
from loraforge.data import load_dataset
from loraforge.modeling import load_quantized_base
from loraforge.qlora import attach_lora
from loraforge.training import train_qlora

config = default_config()
bundle = load_dataset(allow_test=False, config=config.data)
assert bundle.test is None

base_model, tokenizer = load_quantized_base(config)
model = attach_lora(base_model, config)
report = train_qlora(model, tokenizer, bundle, config, root=Path('.'))
print(json.dumps({
    'selected_epoch': report['selection']['selected_epoch'],
    'epoch_macro_f1': {entry['epoch']: entry['validation']['macro_f1'] for entry in report['epochs']},
    'base_validation_macro_f1': report['base_validation_metrics']['macro_f1'],
    'wall_time_seconds': report['wall_time_seconds'],
    'peak_cuda_memory_gib': report['peak_cuda_memory_gib'],
}, indent=2))


## Step 5a — freeze the selection before test exists

This is GPU-free. It recomputes every validation number from the saved logits, re-derives the
winning epoch from the rule, checks the adapter hashes, and fits a **separate** temperature for
the base and tuned systems on validation only. It refuses to run twice.


In [ ]:
!python -m loraforge.cli freeze-selection --root .
!python -m loraforge.cli verify --root .


# STOP — send the evidence for review

Do not restart the runtime and do not evaluate the publisher test yet. Send the complete output from the freeze-and-verify cell for review. The final test belongs in a separate guarded run only after this evidence passes.